# FusionGuard AI - Google Colab Environment
This notebook was auto-generated to run the FusionGuard AI project.

In [2]:
!pip install "fastapi>=0.104.0" "uvicorn[standard]>=0.23.2" "pydantic>=2.4.2" "pydantic-settings>=2.0.3" "torch>=2.1.0" "torchvision>=0.16.0" "transformers>=4.35.0" "opencv-python-headless>=4.8.1" "insightface>=0.7.3" "onnxruntime" "numpy<2.0.0" "python-multipart>=0.0.6" "httpx>=0.25.0" pyngrok nest-asyncio

In [3]:
!mkdir -p app/utils app app/services app/api app/models
!mkdir -p data

In [4]:
%%writefile app/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    PROJECT_NAME: str = "FusionGuard AI"
    VERSION: str = "1.0.0"

    # Model settings
    FACE_RECOGNITION_THRESHOLD: float = 0.5
    EMBEDDINGS_DIR: str = "data/embeddings"

    # Default Policy Rules if none provided in request
    DEFAULT_ALLOWED_ACTIVITIES: list[str] = [
        'a person working on a computer',
        'a person writing on a whiteboard',
        'a person reading a book',
        'a group of students discussing',
        'a person conducting an experiment',
    ]
    DEFAULT_UNAUTHORIZED_ACTIVITIES: list[str] = [
        'a person taking photos of equipment',
        'a person eating food in the lab',
        'a person sleeping at the desk',
        'an unauthorized person in the lab',
        'a person tampering with equipment',
    ]

    class Config:
        env_file = ".env"

settings = Settings()


Writing app/config.py


In [5]:
%%writefile app/__init__.py


Writing app/__init__.py


In [6]:
%%writefile app/main.py
from fastapi import FastAPI
from app.api.routes import router
from app.config import settings

app = FastAPI(
    title=settings.PROJECT_NAME,
    version=settings.VERSION,
    description="Multimodal Context-Aware Surveillance System API"
)

app.include_router(router, prefix="/api/v1")

@app.get("/health")
def health_check():
    return {"status": "healthy", "version": settings.VERSION}


Writing app/main.py


In [7]:
%%writefile app/utils/__init__.py


Writing app/utils/__init__.py


In [8]:
%%writefile app/models/__init__.py


Writing app/models/__init__.py


In [9]:
%%writefile app/models/schemas.py
from pydantic import BaseModel, Field
from typing import List, Optional

class IdentityResult(BaseModel):
    identity: str = Field(description="Recognized identity or 'UNKNOWN'")
    confidence: float = Field(description="Confidence score of recognition")

class SceneResult(BaseModel):
    caption: str = Field(description="Generated description of the scene")

class ActivityResult(BaseModel):
    activity: str = Field(description="Closest matching activity description")
    status: str = Field(description="'AUTHORIZED' or 'UNAUTHORIZED'")
    confidence: float = Field(description="Confidence score of the classification")

class FusionDecision(BaseModel):
    alert_level: str = Field(description="Global threat level: 'GREEN', 'YELLOW', 'RED'")
    message: str = Field(description="Justification for the decision")

class AnalysisResponse(BaseModel):
    identity: IdentityResult
    scene: SceneResult
    activity: ActivityResult
    decision: FusionDecision

class PolicyRules(BaseModel):
    allowed_activities: Optional[List[str]] = None
    unauthorized_activities: Optional[List[str]] = None


Writing app/models/schemas.py


In [10]:
%%writefile app/api/__init__.py


Writing app/api/__init__.py


In [11]:
%%writefile app/api/routes.py
from fastapi import APIRouter, File, UploadFile, Depends, Form
from typing import Optional, List
import numpy as np
import cv2
import json

from app.models.schemas import AnalysisResponse, PolicyRules
from app.services.identity_service import IdentityService
from app.services.scene_service import SceneService
from app.services.activity_service import ActivityService
from app.services.fusion_service import FusionService
from app.config import settings

router = APIRouter()

# Dependency injection
def get_identity_service(): return IdentityService()
def get_scene_service(): return SceneService()
def get_activity_service(): return ActivityService()
def get_fusion_service(): return FusionService()

@router.post("/enroll")
async def enroll_identity(
    identity: str = Form(...),
    file: UploadFile = File(...),
    identity_service: IdentityService = Depends(get_identity_service)
):
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    success = identity_service.enroll(image, identity)
    if success:
        return {"status": "success", "message": f"Successfully enrolled identity: {identity}"}
    return {"status": "error", "message": "No face detected in the image"}

@router.post("/analyze", response_model=AnalysisResponse)
async def analyze_scene(
    file: UploadFile = File(...),
    allowed_activities: Optional[str] = Form(None),
    unauthorized_activities: Optional[str] = Form(None),
    identity_service: IdentityService = Depends(get_identity_service),
    scene_service: SceneService = Depends(get_scene_service),
    activity_service: ActivityService = Depends(get_activity_service),
    fusion_service: FusionService = Depends(get_fusion_service)
):
    # Process image
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Decode JSON rules if provided
    allowed = settings.DEFAULT_ALLOWED_ACTIVITIES
    if allowed_activities and allowed_activities.strip():
        try:
            allowed = json.loads(allowed_activities)
        except json.JSONDecodeError:
            allowed = [x.strip() for x in allowed_activities.split(",")]

    unauthorized = settings.DEFAULT_UNAUTHORIZED_ACTIVITIES
    if unauthorized_activities and unauthorized_activities.strip():
        try:
            unauthorized = json.loads(unauthorized_activities)
        except json.JSONDecodeError:
            unauthorized = [x.strip() for x in unauthorized_activities.split(",")]

    # 1. Pipeline: Identity
    identity_result = identity_service.analyze(image)

    # 2. Pipeline: Scene
    scene_result = scene_service.analyze(image)

    # 3. Pipeline: Activity
    activity_result = activity_service.analyze(
        image=image,
        caption=scene_result.caption,
        allowed_activities=allowed,
        unauthorized_activities=unauthorized
    )

    # 4. Pipeline: Fusion
    decision_result = fusion_service.evaluate(
        identity=identity_result,
        scene=scene_result,
        activity=activity_result
    )

    return AnalysisResponse(
        identity=identity_result,
        scene=scene_result,
        activity=activity_result,
        decision=decision_result
    )


Writing app/api/routes.py


In [12]:
%%writefile app/services/fusion_service.py
from app.models.schemas import IdentityResult, SceneResult, ActivityResult, FusionDecision

class FusionService:
    """
    FusionService implements the multimodal 'late fusion' logic for the system.
    It aggregates results from Identity, Scene, and Activity services to
    produce a final security decision (RED, YELLOW, or GREEN alert).
    """
    def evaluate(self,
                 identity: IdentityResult,
                 scene: SceneResult,
                 activity: ActivityResult) -> FusionDecision:
        """
        Combines multiple AI signals into a single actionable security decision.

        Logic Overview:
        - GREEN: Known person performing an authorized task.
        - YELLOW: Known person performing an unauthorized task (Policy violation).
        - RED: Unknown person or unidentified face (Security breach).

        Args:
            identity (IdentityResult): The output from IdentityService (WHO).
            scene (SceneResult): The output from SceneService (CONTEXT).
            activity (ActivityResult): The output from ActivityService (INTENT).

        Returns:
            FusionDecision: The final alert level and a descriptive message.
        """

        # Rule-based heuristic decision logic

        # CASE 1: Identity is verified and activity matches 'allowed' list.
        if identity.identity != "UNKNOWN" and activity.status == "AUTHORIZED":
            return FusionDecision(
                alert_level="GREEN",
                message=f"Safe: {identity.identity} recognized performing authorized activity: {activity.activity}."
            )

        # CASE 2: Identity is verified, but the activity is in the 'unauthorized' list.
        # This represents a known individual doing something they aren't supposed to.
        elif identity.identity != "UNKNOWN" and activity.status == "UNAUTHORIZED":
            return FusionDecision(
                alert_level="YELLOW",
                message=f"Warning: Known entity ({identity.identity}) performing unauthorized activity: {activity.activity}."
            )

        # CASE 3: Default fallback.
        # If the face is UNKNOWN or NO_FACE_DETECTED, we trigger a high-level alert.
        else:
            return FusionDecision(
                alert_level="RED",
                message=f"Alert: Unknown or unauthorized entity detected. Scene: {scene.caption}."
            )


Writing app/services/fusion_service.py


In [13]:
%%writefile app/services/__init__.py


Writing app/services/__init__.py


In [14]:
%%writefile app/services/identity_service.py
import os
import numpy as np
import pickle
import logging
from insightface.app import FaceAnalysis
from app.models.schemas import IdentityResult
from app.config import settings
from numpy.linalg import norm

logger = logging.getLogger(__name__)

class IdentityService:
    """
    IdentityService handles face detection and recognition using the InsightFace library.
    It manages a local database of face embeddings (PKL files) and provides methods to
    enroll new identities and analyze frames to identify known individuals.
    """
    def __init__(self):
        """
        Initializes the InsightFace FaceAnalysis application with the 'buffalo_l' model.
        Prepares the model for inference on the CPU and loads existing known embeddings.
        """
        logger.info("Initializing IdentityService (InsightFace)...")
        # Initialize InsightFace model
        # buffalo_l is a collection of models for detection, recognition, alignment, etc.
        # Using CPUExecutionProvider for broad compatibility across different environments.
        self.app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
        # Prepare the model: ctx_id=0 (GPU ID, -1 for CPU but InsightFace handles 0 for CPU if provider is CPU)
        # det_size=(640, 640) is the input resolution for the face detector.
        self.app.prepare(ctx_id=0, det_size=(640, 640))

        # Directory where face embeddings (.pkl) are stored locally
        self.embeddings_dir = settings.EMBEDDINGS_DIR
        os.makedirs(self.embeddings_dir, exist_ok=True)
        # In-memory cache of identity:embedding mappings
        self.known_embeddings = self._load_known_embeddings()

    def _load_known_embeddings(self) -> dict:
        """
        Loads all stored face embeddings from the data directory into memory.

        Returns:
            dict: A dictionary mapping identity names to their 512-d feature vectors.
        """
        known = {}
        for filename in os.listdir(self.embeddings_dir):
            if filename.endswith(".pkl"):
                identity = os.path.splitext(filename)[0]
                with open(os.path.join(self.embeddings_dir, filename), "rb") as f:
                    known[identity] = pickle.load(f)
        return known

    def enroll(self, image: np.ndarray, identity: str) -> bool:
        """
        Extracts a face embedding from the provided image and saves it as a new identity.

        Args:
            image (np.ndarray): The input image (BGR format).
            identity (str): The unique name/ID for this person.

        Returns:
            bool: True if enrollment was successful (face found), False otherwise.
        """
        # Detect faces and extract features
        faces = self.app.get(image)
        if not faces:
            logger.warning(f"Enrollment failed: No face detected for {identity}")
            return False

        # Take the most prominent face (usually faces[0] in InsightFace sorted by detection score)
        embedding = faces[0].normed_embedding

        # Persist embedding to disk
        file_path = os.path.join(self.embeddings_dir, f"{identity}.pkl")
        with open(file_path, "wb") as f:
            pickle.dump(embedding, f)

        # Update in-memory cache
        self.known_embeddings[identity] = embedding
        logger.info(f"Successfully enrolled identity: {identity}")
        return True

    def analyze(self, image: np.ndarray) -> IdentityResult:
        """
        Detects the primary face in an image and compares it against enrolled identities.
        Uses cosine similarity between 512-dimensional embeddings.

        Args:
            image (np.ndarray): The input image to analyze.

        Returns:
            IdentityResult: Contains the matched identity name and the confidence score.
        """
        faces = self.app.get(image)
        if not faces:
            return IdentityResult(identity="NO_FACE_DETECTED", confidence=0.0)

        # Extract the embedding for the detected face
        target_embedding = faces[0].normed_embedding

        best_match = "UNKNOWN"
        best_score = 0.0

        # Compare target embedding with all known embeddings in the database
        for identity, known_embedding in self.known_embeddings.items():
            # InsightFace embeddings are normalized, so dot product equals cosine similarity.
            # Range is generally [-1, 1], with > 0.4 usually being a strong match for ArcFace.
            similarity = np.dot(target_embedding, known_embedding)
            if similarity > best_score:
                best_score = similarity
                best_match = identity

        # Validate against configured threshold
        if best_score >= settings.FACE_RECOGNITION_THRESHOLD:
            return IdentityResult(identity=best_match, confidence=float(best_score))

        return IdentityResult(identity="UNKNOWN", confidence=float(best_score))


Writing app/services/identity_service.py


In [15]:
%%writefile app/services/scene_service.py
import numpy as np
import logging
from PIL import Image
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from app.models.schemas import SceneResult

logger = logging.getLogger(__name__)

class SceneService:
    """
    SceneService generates natural language descriptions (captions) from input images.
    It utilizes the BLIP (Bootstrapping Language-Image Pre-training) model from Salesforce
    to perform image-to-text generation.
    """
    def __init__(self):
        """
        Initializes the BLIP processor and model.
        Automatically detects and utilizes the best available hardware (CUDA, MPS, or CPU).
        """
        logger.info("Initializing SceneService (BLIP-base)...")
        # Device priority: NVIDIA GPU (CUDA) > Apple Silicon (MPS) > Standard CPU
        self.device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

        # Load the base BLIP model which is optimized for standard image captioning tasks.
        self.processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(self.device)

    def analyze(self, image: np.ndarray) -> SceneResult:
        """
        Generates a descriptive caption for a given image frame.

        Args:
            image (np.ndarray): The input image frame (BGR format from OpenCV).

        Returns:
            SceneResult: Encapsulated string caption describing the scene.
        """
        # Convert OpenCV image (BGR numpy array) to PIL Image (RGB) as expected by Transformers
        rgb_image = image[:, :, ::-1]
        pil_image = Image.fromarray(rgb_image)

        # Preprocess the image and move tensors to the active device (GPU/CPU)
        inputs = self.processor(pil_image, return_tensors="pt").to(self.device)

        # Generate the caption using beam search or greedy decoding (default)
        # max_new_tokens=50 ensures the description stays concise but informative.
        out = self.model.generate(**inputs, max_new_tokens=500)

        # Decode the generated tokens back into a human-readable string
        caption = self.processor.decode(out[0], skip_special_tokens=True)

        logger.info(f"Generated scene caption: {caption}")
        return SceneResult(caption=caption)


Writing app/services/scene_service.py


In [16]:
%%writefile app/services/activity_service.py
import numpy as np
import logging
from typing import Optional, List
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from app.models.schemas import ActivityResult

logger = logging.getLogger(__name__)

class ActivityService:
    """
    ActivityService performs zero-shot classification to verify if an ongoing
    scene activity is authorized or unauthorized.
    It leverages the OpenAI CLIP (Contrastive Language-Image Pre-training) model
    which can compare images against text descriptions directly or compare
    text captions against a set of predefined labels (text-to-text matching).
    """
    def __init__(self):
        """
        Initializes the CLIP model and processor.
        Detects GPU (CUDA/MPS) or falls back to CPU.
        """
        logger.info("Initializing ActivityService (CLIP)...")
        self.device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        self.model_id = "openai/clip-vit-base-patch32"
        self.processor = CLIPProcessor.from_pretrained(self.model_id)
        self.model = CLIPModel.from_pretrained(self.model_id).to(self.device)

    def analyze(self,
                image: Optional[np.ndarray] = None,
                caption: Optional[str] = None,
                allowed_activities: Optional[List[str]] = None,
                unauthorized_activities: Optional[List[str]] = None) -> ActivityResult:
        """
        Determines the most likely activity based on image content OR text caption.
        Uses CLIP to find the highest similarity match from the provided activity lists.

        Args:
            image (Optional[np.ndarray]): The raw BGR image array (optional).
            caption (Optional[str]): A text description of the scene (optional).
            allowed_activities (Optional[List[str]]): List of permissible activities.
            unauthorized_activities (Optional[List[str]]): List of forbidden activities.

        Returns:
            ActivityResult: Includes the identified activity label, its authorization status, and confidence score.
        """
        allowed = allowed_activities or []
        unauthorized = unauthorized_activities or []
        all_activities = allowed + unauthorized

        # If no policy rules are defined, we cannot classify the intent.
        if not all_activities:
            logger.warning("No activities provided for classification.")
            return ActivityResult(activity="None defined", status="UNAUTHORIZED", confidence=0.0)

        probs = []
        # BRANCH 1: Visual Classification (CLIP Image-Text comparison)
        if image is not None:
            # Prepare image for CLIP
            rgb_image = image[:, :, ::-1]
            pil_image = Image.fromarray(rgb_image)

            # CLIP finds which text prompt best matches the visual features of the image.
            inputs = self.processor(text=all_activities, images=pil_image, return_tensors="pt", padding=True).to(self.device)
            outputs = self.model(**inputs)

            # Extract probability distribution across the activity labels.
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1).cpu().detach().numpy()[0]

        # BRANCH 2: Semantic Verification (CLIP Text-Text comparison)
        # Useful if image isn't available or we want to verify the caption generated by SceneService.
        elif caption is not None:
            # Generate shared embeddings for the caption and all possible activity labels.
            inputs = self.processor(text=[caption] + all_activities, return_tensors="pt", padding=True).to(self.device)
            text_features = self.model.get_text_features(**inputs)
            # Normalize embeddings to calculate cosine similarity via dot product.
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            caption_embed = text_features[0]
            rules_embeds = text_features[1:]

            # Calculate similarity scores between the caption and each rule.
            similarities = (rules_embeds @ caption_embed.T).squeeze(0)
            probs = similarities.softmax(dim=0).cpu().detach().numpy()

        else:
            return ActivityResult(activity="No input", status="UNAUTHORIZED", confidence=0.0)

        # Map back to the best fitting activity label
        best_idx = np.argmax(probs)
        best_activity = all_activities[best_idx]
        confidence = float(probs[best_idx])

        # Determine final authorization status based on grouping.
        status = "AUTHORIZED" if best_activity in allowed else "UNAUTHORIZED"

        logger.info(f"Classified activity as: {best_activity} ({status}) with {confidence:.2f} confidence.")
        return ActivityResult(
            activity=best_activity,
            status=status,
            confidence=confidence
        )


Writing app/services/activity_service.py


In [17]:
%%writefile requirements.txt
fastapi>=0.104.0
uvicorn[standard]>=0.23.2
pydantic>=2.4.2
pydantic-settings>=2.0.3
torch>=2.1.0
torchvision>=0.16.0
transformers>=4.35.0
opencv-python-headless>=4.8.1
insightface>=0.7.3
onnxruntime
numpy<2.0.0
python-multipart>=0.0.6
httpx>=0.25.0


Overwriting requirements.txt


In [18]:
%%writefile test_stubs.py
from fastapi.testclient import TestClient
from app.main import app
import numpy as np
import cv2
import json

client = TestClient(app)

# Create a dummy image
img = np.zeros((100, 100, 3), dtype=np.uint8)
cv2.imwrite("data/test_images/dummy.jpg", img)

print("Pinging analyze endpoint...")
try:
    with open("data/test_images/dummy.jpg", "rb") as f:
        files = {"file": ("dummy.jpg", f, "image/jpeg")}
        data = {
            "allowed_activities": json.dumps(["walking"]),
            "unauthorized_activities": json.dumps(["running"])
        }
        response = client.post("/api/v1/analyze", files=files, data=data)

    print("Status Code:", response.status_code)
    try:
        print("Response payload:", json.dumps(response.json(), indent=2))
    except json.JSONDecodeError:
        print("Response payload:", response.text)
except Exception as e:
    print(f"Failed to connect: {e}")


Writing test_stubs.py


In [7]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio
from google.colab import userdata
from app.main import app  # Assuming this is your FastAPI app

# 1. Setup Environment
nest_asyncio.apply()

# 2. Setup Ngrok Auth (With proper error handling)
try:
    # Make sure the secret name here exactly matches the name in your Colab Secrets tab
    NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ Ngrok Auth Token loaded successfully.")
except userdata.SecretNotFoundError:
    print("❌ ERROR: 'NGROK_TOKEN' not found in Colab Secrets.")
    print("Please add it via the 🔑 icon on the left sidebar.")
    # Optional fallback for testing:
    # ngrok.set_auth_token("paste_your_token_here_temporarily")
except Exception as e:
    print(f"❌ Ngrok Auth Error: {e}")

# 3. Setup Tunnel
try:
    # Clean up existing tunnels to prevent conflicts
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)

    # Create the new tunnel (explicitly binding to http)
    ngrok_tunnel = ngrok.connect(8000, bind_tls=True)
    print(f'🚀 Public URL: {ngrok_tunnel.public_url}')
except Exception as e:
    print(f"❌ Ngrok connection error: {e}")

# 4. Run Server in Background
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
# Prevent starting multiple server tasks in the same loop
if not any("serve" in getattr(t.get_coro(), "__name__", "") for t in asyncio.all_tasks()):
    loop.create_task(server.serve())
    print("✅ Server task started successfully in the background.")
else:
    print("ℹ️ Server is already running.")

✅ Ngrok Auth Token loaded successfully.


🚀 Public URL: https://autocatalytically-unredeemable-regena.ngrok-free.dev
ℹ️ Server is already running.


In [4]:
import httpx
import asyncio

async def check_local_health():
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get('http://127.0.0.1:8000/health')
            print(f'Local Server Status: {response.json()}')
        except Exception as e:
            print(f'Server not reachable locally yet: {e}')

# Run the health check
await check_local_health()

INFO:     127.0.0.1:40920 - "GET /health HTTP/1.1" 200 OK
Local Server Status: {'status': 'healthy', 'version': '1.0.0'}


In [5]:
import httpx
import cv2
import numpy as np
import json

# Create a test image (black square)
test_img = np.zeros((300, 300, 3), dtype=np.uint8)
cv2.imwrite('test_input.jpg', test_img)

async def test_analyze_endpoint():
    url = 'http://127.0.0.1:8000/api/v1/analyze'
    files = {'file': ('test.jpg', open('test_input.jpg', 'rb'), 'image/jpeg')}
    data = {
        'allowed_activities': json.dumps(['a person sitting']),
        'unauthorized_activities': json.dumps(['a person running'])
    }

    async with httpx.AsyncClient() as client:
        try:
            # Model initialization happens on the first request, so we use a long timeout
            response = await client.post(url, files=files, data=data, timeout=120.0)
            print(f'Status: {response.status_code}')
            print('Analysis Result:', json.dumps(response.json(), indent=2))
        except Exception as e:
            print(f'Test failed: {e}')

await test_analyze_endpoint()

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:06<00:00, 44674.24KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

INFO:     127.0.0.1:56756 - "POST /api/v1/analyze HTTP/1.1" 200 OK
Status: 200
Analysis Result: {
  "identity": {
    "identity": "NO_FACE_DETECTED",
    "confidence": 0.0
  },
  "scene": {
    "caption": "a black background with a white and red flower"
  },
  "activity": {
    "activity": "a person running",
    "status": "UNAUTHORIZED",
    "confidence": 0.6486328840255737
  },
  "decision": {
    "alert_level": "YELLOW",
    "message": "Warning: Known entity (NO_FACE_DETECTED) performing unauthorized activity: a person running."
  }
}


### Troubleshooting Tip
If the local health check fails, try re-running the `run_server` cell. If it passes but the Public URL still shows 'Offline', give it another minute—the heavy AI models (especially BLIP/CLIP) take time to initialize on the first request.